# Fusion Evaluation Notebook - Games

Test the updated pipeline evaluation with automatic ID alignment via `_fusion_sources`.

In [1]:
import pandas as pd
import json
from pathlib import Path

# Add PyDI to path
import sys
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

## 1. Load Data

In [2]:
# Configuration - update these paths
OUTPUT_DIR = Path(".")
FUSION_DIR = OUTPUT_DIR / "fusion"
TEST_DIR = Path("../../../usecases/input/games/fusion")

# Load fused data (from best case or specify path)
best_case_path = FUSION_DIR / "optimization" / "best_case.json"
if best_case_path.exists():
    with open(best_case_path) as f:
        best_info = json.load(f)
    best_case_dir = Path(best_info["best_case_dir"])
    print(f"Best case: {best_info['best_case_key']} (accuracy: {best_info['best_accuracy']:.1%})")
else:
    # Fallback to fused_clean.csv
    best_case_dir = FUSION_DIR

# Load fused data
fused_path = best_case_dir / "fused.csv"
if not fused_path.exists():
    fused_path = FUSION_DIR / "fused_clean.csv"
    
fused_df = pd.read_csv(fused_path)
print(f"Loaded fused data: {len(fused_df)} rows from {fused_path}")
fused_df.head()

Best case: iterative__opt_llm_omit (accuracy: 30.3%)
Loaded fused data: 61282 rows from fusion/fused_clean.csv


,_id,_fusion_sources,series,id,name,releaseYear,userScore,platform,criticScore,developer,ESRB,publisher,globalSales
0,metacritic_12711,"['metacritic_12711', 'dbpedia_41840']",NaN,metacritic_12711,Magic: The Gathering - Duels of the Planeswalkers,2010-01-01,4.9,Windows PC,69.0,Stainless Games,T,NaN,NaN
1,dbpedia_8431,"['dbpedia_8431', 'metacritic_1377', 'dbpedia_5...",Minecraft (franchise),dbpedia_8431,Minecraft,2018-01-01,7.4,Nintendo Switch,86.0,Mojang Studios,E10+,NaN,NaN
2,metacritic_1650,"['metacritic_1650', 'sales_1799', 'dbpedia_350...",Dragon Age,metacritic_1650,Dragon Age: Inquisition,2014-01-01,6.0,Windows PC,85.0,BioWare,M,Electronic Arts,0.0
3,metacritic_9626,"['metacritic_9626', 'sales_787', 'metacritic_9...",NaN,metacritic_9626,LEGO Pirates of the Caribbean: The Video Game,2011-01-01,7.3,Xbox 360,73.0,Traveller's Tales,E10+,Disney Interactive Studios,1.0
4,dbpedia_14385,"['dbpedia_14385', 'dbpedia_4031', 'metacritic_...",NaN,dbpedia_14385,Death Jr.,2005-01-01,7.6,PlayStation Portable (PSP),64.5,Backbone Entertainment,T,Konami Digital Entertainment,0.0


In [3]:
# Load test set
from PyDI.io.loaders import load_xml
from PyDI.normalization import load_normalization_spec
from PyDI.normalization.transform import transform_dataframe

test_xml = TEST_DIR / "test_set.xml"
if test_xml.exists():
    test_df = load_xml(test_xml, nested_handling="aggregate")
    print(f"Loaded test set: {len(test_df)} rows from {test_xml}")
else:
    # Try CSV
    test_csv = TEST_DIR / "test_set.csv"
    test_df = pd.read_csv(test_csv)
    print(f"Loaded test set: {len(test_df)} rows from {test_csv}")

# Apply normalization (same as pipeline Step 1)
SCHEMA_PATH = Path("../../../usecases/input/games/schemamatching/target_schema.json")
SCHEMA_MATCHING_DIR = OUTPUT_DIR / "schema_matching"  # For taxonomy cache

with open(SCHEMA_PATH) as f:
    target_schema = json.load(f)

spec = load_normalization_spec(target_schema)
result = transform_dataframe(
    test_df,
    spec,
    chat_model=None,  # Use cached taxonomy mappings only
    taxonomy_cache_dir=str(SCHEMA_MATCHING_DIR),
    schema_base_path=str(SCHEMA_PATH.parent),
)
test_df = result.dataframe
print(f"Normalized test set: {len(test_df)} records")

test_df.head()

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Column 'globalSales' not found in DataFrame
Column 'series' not found in DataFrame


Loaded test set: 15 rows from ../../../usecases/input/games/fusion/test_set.xml

  Taxonomy mapping for 'test_set.platform':
    Total unique values: 6
    Already in taxonomy: 1
    Found in cache:      5
    Need LLM mapping:    0
    All values already mapped!
Normalized test set: 15 records


,id,name,releaseYear,developer,genres_genre,publisher,platform,criticScore,userScore,ESRB
0,metacritic_17606,Army of Two: The Devil's Cartel,2013-01-01,Visceral Games,"[Action, Krimi]",Electronic Arts,PlayStation 3,58.0,6.2,AO
1,metacritic_3124,Lego Marvel Super Heroes,2013-01-01,Traveller's Tales,"[Action, Adventure, Fantasy, Science-Fiction]",Warner Bros. Interactive Entertainment,PlayStation 3,82.0,8.0,E10+
2,metacritic_12057,SingStar ABBA,2008-01-01,London Studio,Music,SCEA,PlayStation 3,70.0,NaN,T
3,metacritic_4212,Tiger Woods PGA Tour 10,2009-01-01,EA Tiburon,Sports,Electronic Arts,PlayStation 3,80.0,4.8,E
4,metacritic_5860,Dynasty Warriors 3,2001-01-01,Omega Force,Hack and slash,Koei,PlayStation 2,78.0,7.8,T


## 2. Inspect IDs

In [4]:
print("Fused IDs (first 20):")
print(fused_df["id"].head(20).tolist())
print(f"\nFused ID dtype: {fused_df['id'].dtype}")
print(f"Fused ID nulls: {fused_df['id'].isna().sum()}")

Fused IDs (first 20):
['metacritic_12711', 'dbpedia_8431', 'metacritic_1650', 'metacritic_9626', 'dbpedia_14385', 'dbpedia_15290', 'sales_970', 'dbpedia_47266', 'dbpedia_64389', 'dbpedia_35436', 'dbpedia_47666', 'metacritic_10749', 'dbpedia_38608', 'metacritic_3987', 'metacritic_19142', 'metacritic_15739', 'sales_6784', 'dbpedia_26686', 'metacritic_9685', 'sales_2202']

Fused ID dtype: object
Fused ID nulls: 0


In [5]:
print("Test set IDs (first 20):")
print(test_df["id"].head(20).tolist())
print(f"\nTest ID dtype: {test_df['id'].dtype}")
print(f"Test ID nulls: {test_df['id'].isna().sum()}")

Test set IDs (first 20):
['metacritic_17606', 'metacritic_3124', 'metacritic_12057', 'metacritic_4212', 'metacritic_5860', 'metacritic_9221', 'metacritic_8878', 'metacritic_8567', 'metacritic_14340', 'metacritic_5286', 'metacritic_19481', 'metacritic_14164', 'metacritic_17161', 'metacritic_9865', 'metacritic_19234']

Test ID dtype: object
Test ID nulls: 0


In [6]:
# Check overlap
fused_ids = set(fused_df["id"].dropna().astype(str))
test_ids = set(test_df["id"].dropna().astype(str))

overlap = fused_ids & test_ids
print(f"Fused IDs: {len(fused_ids)}")
print(f"Test IDs: {len(test_ids)}")
print(f"Overlap: {len(overlap)}")
print(f"\nSample fused IDs not in test: {list(fused_ids - test_ids)[:5]}")
print(f"Sample test IDs not in fused: {list(test_ids - fused_ids)[:5]}")

Fused IDs: 61282
Test IDs: 15
Overlap: 5

Sample fused IDs not in test: ['dbpedia_13435', 'dbpedia_51331', 'dbpedia_40480', 'metacritic_267', 'dbpedia_13193']
Sample test IDs not in fused: ['metacritic_4212', 'metacritic_12057', 'metacritic_9221', 'metacritic_8878', 'metacritic_19234']


## 3. Test Pipeline Evaluation (with automatic ID alignment)

The pipeline's `DataFusionEvaluator` now automatically handles ID alignment by indexing all IDs in `_fusion_sources`.

In [7]:
from PyDI.fusion import DataFusionStrategy, DataFusionEvaluator
from PyDI.fusion.evaluation import tokenized_match, year_only_match, set_equality_match
from PyDI.pipeline.fusion_optimization import _numeric_tolerance_match_relative
from functools import partial

# Create a strategy with type-aware evaluation functions
strategy = DataFusionStrategy("test_evaluation")

# Define attribute types based on the test data
attr_types = {
    "name": "string",
    "releaseYear": "date", 
    "developer": "string",
    "publisher": "string",
    "platform": "string",
    "criticScore": "numeric",
    "userScore": "numeric",
    "ESRB": "string",
}

# Add evaluation functions based on type
for attr, attr_type in attr_types.items():
    if attr_type == "date":
        strategy.add_evaluation_function(attr, year_only_match)
    elif attr_type == "numeric":
        strategy.add_evaluation_function(attr, partial(_numeric_tolerance_match_relative, tolerance=0.2))
    else:
        strategy.add_evaluation_function(attr, tokenized_match)

# Create evaluator with debug output
evaluator = DataFusionEvaluator(
    strategy,
    debug=True,
    debug_file=FUSION_DIR / "pipeline_eval_debug.jsonl",
)

# Run evaluation - this now uses automatic ID alignment via _fusion_sources
results = evaluator.evaluate(
    fused_df=fused_df,
    fused_id_column="id",
    expected_df=test_df,
    expected_id_column="id",
)

print(f"\n{'='*60}")
print(f"PIPELINE EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Overall accuracy: {results['overall_accuracy']:.1%}")
print(f"Total evaluations: {results['total_evaluations']}")
print(f"Correct: {results['total_correct']}")
print(f"Records evaluated: {results['num_evaluated_records']}")


PIPELINE EVALUATION RESULTS
Overall accuracy: 88.2%
Total evaluations: 119
Correct: 105
Records evaluated: 15


In [8]:
# Per-attribute breakdown from pipeline evaluation
print("\nPer-attribute accuracy (from pipeline):")
print("-" * 50)
for key, value in sorted(results.items()):
    if key.endswith("_accuracy") and key not in ("overall_accuracy", "macro_accuracy"):
        attr = key.replace("_accuracy", "")
        count = results.get(f"{attr}_count", 0)
        correct = int(value * count) if count > 0 else 0
        print(f"{attr:30s}: {correct:3d}/{count:3d} = {value:.1%}")


Per-attribute accuracy (from pipeline):
--------------------------------------------------
ESRB                          :  13/ 15 = 86.7%
criticScore                   :  15/ 15 = 100.0%
developer                     :  10/ 15 = 66.7%
name                          :  15/ 15 = 100.0%
platform                      :  15/ 15 = 100.0%
publisher                     :  11/ 15 = 73.3%
releaseYear                   :  15/ 15 = 100.0%
userScore                     :  11/ 14 = 78.6%


## 4. View Debug Log (Mismatches)

The debug log shows details of each mismatch including the conflict resolution rule used.

In [9]:
# Read debug log to see mismatches
debug_path = FUSION_DIR / "pipeline_eval_debug.jsonl"
mismatches = []

if debug_path.exists():
    with open(debug_path) as f:
        for line in f:
            entry = json.loads(line)
            if entry.get("type") == "evaluation_mismatch":
                mismatches.append({
                    "attribute": entry.get("attribute"),
                    "fused_id": entry.get("fused_id"),
                    "expected": entry.get("expected_value"),
                    "fused": entry.get("fused_value"),
                    "conflict_rule": entry.get("conflict_rule"),
                    "eval_rule": entry.get("evaluation_rule"),
                    "reason": entry.get("reason"),
                })

print(f"Mismatches: {len(mismatches)}")
if mismatches:
    mismatch_df = pd.DataFrame(mismatches)
    display(mismatch_df)

Mismatches: 14


,attribute,fused_id,expected,fused,conflict_rule,eval_rule,reason
0,publisher,dbpedia_40616,Warner Bros. Interactive Entertainment,None,default,tokenized_match,missing_fused_value
1,publisher,sales_1430,SCEA,Sony Computer Entertainment,default,tokenized_match,mismatch
2,publisher,metacritic_5860,Koei,THQ,default,tokenized_match,mismatch
3,publisher,dbpedia_42608,Activision,None,default,tokenized_match,missing_fused_value
4,userScore,dbpedia_42162,4.8,7.55,default,_numeric_tolerance_match_relative,mismatch
5,userScore,metacritic_14340,7.1,4.3,default,_numeric_tolerance_match_relative,mismatch
6,userScore,dbpedia_57362,4.2,5.85,default,_numeric_tolerance_match_relative,mismatch
7,ESRB,sales_3725,AO,M,default,tokenized_match,mismatch
8,ESRB,metacritic_19481,M17+,M,default,tokenized_match,mismatch
9,developer,dbpedia_40616,Traveller's Tales,TT Fusion,default,tokenized_match,mismatch


## 5. Export Results

In [10]:
# Save mismatches for review
if mismatches:
    mismatch_path = FUSION_DIR / "pipeline_eval_mismatches.csv"
    pd.DataFrame(mismatches).to_csv(mismatch_path, index=False)
    print(f"Saved mismatches to {mismatch_path}")

# Save summary using pipeline results format
summary = {
    "overall_accuracy": results.get("overall_accuracy", 0),
    "total_evaluations": results.get("total_evaluations", 0),
    "total_correct": results.get("total_correct", 0),
    "num_evaluated_records": results.get("num_evaluated_records", 0),
    "per_attribute": {
        key.replace("_accuracy", ""): {
            "accuracy": value,
            "count": results.get(f"{key.replace('_accuracy', '')}_count", 0),
        }
        for key, value in results.items()
        if key.endswith("_accuracy") and key not in ("overall_accuracy", "macro_accuracy")
    }
}

summary_path = FUSION_DIR / "pipeline_eval_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved summary to {summary_path}")

Saved mismatches to fusion/pipeline_eval_mismatches.csv
Saved summary to fusion/pipeline_eval_summary.json
